-----
# Data Cleaning
-----

### Notebook Summary

All extracted information will act as features in my dataset and will be used in embeddings, clustering and eventually, for building classification models.

In this notebook, I clean the data by removing punctuation, whitespace, and other irrelevant content. This cleaning step is needed to ensure that, before vectorisation, the data focuses solely on the actual email content, making results more accurate.

## Set Up
---

In [109]:
import numpy as np
import pandas as pd
import re
import string

## Functions
-----

In [110]:
def df_check (df):

    shape = df.shape
    nulls = df.isna().sum().sum()
    duplicated_rows = df.duplicated().sum()
    duplicated_cols = df.columns.duplicated().sum()


    print (f"""      
    Number of Rows: {shape[0]}     
    Number of Columns: {shape[1]}     
    Number of Nulls: {nulls}       
    Number of Duplicated Rows: {duplicated_rows}
    Number of Duplicated Cols: {duplicated_cols}
        """)

## Data Loading

In [111]:
emails_df  = pd.read_csv('../../data/processed_emails.csv', index_col=0)

In [112]:
emails_df.head()

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,NaN,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,"Traveling to have a business meeting takes the fun out of the trip. Especially if you have to prepare a presentation. I would suggest holding the business plan meetings here then take a trip without any formal business meetings. I would even try and get some honest opinions on whether a trip is even desired or necessary.\n\nAs far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is not. Too often the presenter speaks and the others are quiet just waiting for their turn. The meetings might be better if held in a round table discussion format. \n\nMy suggestion for where to go is Austin. Play golf and rent a ski boat and jet ski's. Flying somewhere takes too much time."
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Randy,\n\n Can you send me a schedule of the salary and level of everyone in the \nscheduling group. Plus your thoughts on any changes that need to be made. \n(Patti S for example)\n\nPhillip"
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


In [113]:
df_check(emails_df)

      
    Number of Rows: 517401     
    Number of Columns: 4     
    Number of Nulls: 28333       
    Number of Duplicated Rows: 264855
    Number of Duplicated Cols: 0
        


In [114]:
cleaned_df = emails_df[:50_000].copy()

### Nulls
----

In [115]:
emails_df.isna().sum()

from           0
to          9146
subject    19187
body           0
dtype: int64

#### 1. Null Recipients

In [116]:
emails_df[emails_df['to'].isna()]

from   to                                        subject  \
1230    outlook-migration-team@enron.com  NaN                                    Lee Odonnel   
1231    outlook-migration-team@enron.com  NaN                                    Greg Thorse   
3909           jeff.youngflesh@enron.com  NaN           Lexmark Document/Workflow mgmt intro   
4395               john.arnold@enron.com  NaN                                            NaN   
4442               ann.schmidt@enron.com  NaN                                 Enron Mentions   
...                                  ...  ...                                            ...   
516316   john.phillips@clarionenergy.com  NaN             Corrected Post-AGA NYMEX straddles   
516731             andy.zipper@enron.com  NaN          EDS Arcordia, set up meeting Kolodgie   
516732             andy.zipper@enron.com  NaN                               Red  Meteor etc.   
516846           john.zufferli@enron.com  NaN                                            NaN   
516976             janet.mulero@ubsw.com  NaN  Re: Market Risk Meeting Today - John Zufferli   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

-----
**Comment:**

After looking into the null recipients, cases where the to are missing either they are addressed to email groups or no recipient can be found. Making the decision to remove all cases where recipients are missing, for this project I think it important to work with a complete dataset for more accurate insights, models and results.

In [117]:
cleaned_df.dropna(subset=['to'], inplace=True)

In [118]:
cleaned_df.reset_index(drop=True, inplace=True)

#### 2. Null Subject

In [119]:
emails_df[emails_df['subject'].isna()]

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,NaN,Here is our forecast
3,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Randy,\n\n Can you send me a schedule of the salary and level of everyone in the \nscheduling group. Plus your thoughts on any changes that need to be made. \n(Patti S for example)\n\nPhillip"
6,phillip.allen@enron.com,"david.l.johnson@enron.com, john.shafer@enron.com",NaN,Please cc the following distribution list with updates:\n\nPhillip Allen (pallen@enron.com)\nMike Grigsby (mike.grigsby@enron.com)\nKeith Holst (kholst@enron.com)\nMonique Sanchez\nFrank Ermis\nJohn Lavorato\n\n\nThank you for your help\n\nPhillip Allen
11,phillip.allen@enron.com,stagecoachmama@hotmail.com,NaN,"Lucy,\n\n Here are the rentrolls:\n\n\n\n Open them and save in the rentroll folder. Follow these steps so you don't \nmisplace these files.\n\n 1. Click on Save As\n 2. Click on the drop down triangle under Save in:\n 3. Click on the (C): drive\n 4. Click on the appropriate folder\n 5. Click on Save:\n\nPhillip"
14,phillip.allen@enron.com,david.delainey@enron.com,NaN,"Dave, \n\n Here are the names of the west desk members by category. The origination \nside is very sparse. \n\n\n\n\n\nPhillip"
...,...,...,...,...
516846,john.zufferli@enron.com,NaN,NaN,Conference call with UBS
516853,frank.hayden@enron.com,john.zufferli@enron.com,NaN,Sorry I missed call. My understanding is that it went well.\nLet me know if you need anything\nFrank
517247,john.zufferli@enron.com,majordomo@majordomo.pjm,NaN,unsubscribe pjm-customer-info
517289,john.zufferli@enron.com,john.lavorato@enron.com,NaN,home number is (403) 685-4817


------
**Comment:** 

Making decision to also drop cases where email subject is missing from the dataset. Emails with missing subjects only make up 3% of the dataset and for me its more imporant to have a complete dataset for this project and 3% data loss is not a big sacrifice to me. 

In [120]:
cleaned_df.dropna(subset=['subject'], inplace=True)

In [121]:
cleaned_df.reset_index(drop=True, inplace=True)

#### ReCheck of Nulls

In [122]:
df_check(cleaned_df)

      
    Number of Rows: 45243     
    Number of Columns: 4     
    Number of Nulls: 0       
    Number of Duplicated Rows: 21762
    Number of Duplicated Cols: 0
        


### Duplicated Emails
-----

In [123]:
cleaned_df[cleaned_df.duplicated()]

from                                                                                                      to                                      subject  \
414      phillip.allen@enron.com                                                                                   keith.holst@enron.com  Consolidated positions: Issues & To Do list   
415      phillip.allen@enron.com                                                                                   keith.holst@enron.com  Consolidated positions: Issues & To Do list   
416      phillip.allen@enron.com                                                                                  paula.harris@enron.com                         Re: 2001 Margin Plan   
417      phillip.allen@enron.com                                                                                    ina.rangel@enron.com         Var, Reporting and Resources Meeting   
418      phillip.allen@enron.com                                                                                    pallen70@hotmail.com                                     Westgate   
...                          ...                                                                                                     ...                                          ...   
44951  felecia.acevedo@enron.com                                                       andrea.yowman@enron.com, elspeth.inglis@enron.com                         RE: PRC Demographics   
44952    robert.george@enron.com                                                                                       Michelle Cash@ECT      Composicao do Capital Social da ELEKTRO   
44953  office.chairman@enron.com                                                                           all.enron-worldwide@enron.com              Over $50 -- You made it happen!   
44957      james.grace@enron.com                       michelle.cash@enron.com, travis.mccullough@enron.com, \n\tlance.schuler@enron.com                                          Re:   
44958  sasha.divelbiss@enron.com  gerald.nemec@enron.com, travis.mccullough@enron.com, mark.taylor@enron.com, \n\tjulia.murray@enron.com          Alisha Mahabir - Interview Schedule   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

----
**Comment:**

Making the decision to drop all duplicated emails.

In [124]:
cleaned_df.drop_duplicates(inplace=True)

In [125]:
cleaned_df.reset_index(drop= True, inplace=True)

In [126]:
cleaned_df

from                                                 to                            subject  \
0      phillip.allen@enron.com                            john.lavorato@enron.com                                Re:   
1      phillip.allen@enron.com                             leah.arsdall@enron.com                           Re: test   
2      phillip.allen@enron.com                               greg.piper@enron.com                          Re: Hello   
3      phillip.allen@enron.com                               greg.piper@enron.com                          Re: Hello   
4      phillip.allen@enron.com                           joyce.teixeira@enron.com       Re: PRC review - phone calls   
...                        ...                                                ...                                ...   
23476  michelle.cash@enron.com                             twanda.sweet@enron.com                     FW: Pam Benson   
23477  michelle.cash@enron.com                             twanda.sweet@enron.com                            FW: ECI   
23478  michelle.cash@enron.com                              britt.davis@enron.com          RE: In re Event Resources   
23479  michelle.cash@enron.com  david.oxley@enron.com, kathryn.schultea@enron.com                 RE: List of People   
23480  michelle.cash@enron.com                              diane.goode@enron.com  RE: Question from Amy Fitzpatrick   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

#### ReCheck of Duplicates

In [127]:
df_check(cleaned_df)

      
    Number of Rows: 23481     
    Number of Columns: 4     
    Number of Nulls: 0       
    Number of Duplicated Rows: 0
    Number of Duplicated Cols: 0
        


## Cleaning Email Body and Subject
------

Cleaning email body and subject by removing:

- new line and tab characters
- email headers (like to, from, subject, sent)
- punctuation

In [128]:
from bs4 import BeautifulSoup

def clean_email_with_soup(email):
    """
    Description: 
        Clean email data to address noise such as HTML tags, email addresses, phone numbers, files, etc.

    Input:
        Raw email data for a single email.

    Output:
        Cleaned email data for a single email.
    """
    # Use BeautifulSoup to remove all HTML tags
    if '>' in email or  '<' in email:
        soup = BeautifulSoup(email, 'html.parser')
        email = soup.get_text()
    
    # Remove email headers (forwarded by/original message, from, to, subject, sent)
    email = re.sub(r'[-\s]?(Forwarded by|Original Message)(\s*.*?)?[-\s]+', '', email)
    email = re.sub(r'From:.*?(\n|$)', '', email)
    email = re.sub(r'To:.*?(\n|$)', '', email)
    email = re.sub(r'Subject:.*?(\n|$)', '', email)
    email = re.sub(r'Sent:.*?(\n|$)', '', email)

    # Remove email addresses and phone numbers 
    email = re.sub(r'\S+@\S+', '', email)
    email = re.sub(r'(\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}', '', email)

    # Remove files and filepaths
    email = re.sub(r'\b[\w.-]+(?:\.(?:docx?|xlsx?|pdf|txt|html|zip|rar|png|jpe?g|gif))\b', '', email)
    email = re.sub(r'[\w]*FilePath\S*', '', email)   
    
    # Remove web addresses
    email = re.sub(r'http[s]?://\S+|www\.\S+', '', email)

    # Remove non-alphanumeric symbols except for spaces
    email = re.sub(r'[^\w\s]|_', '', email)
 
    # Remove non-ASCII characters
    email = re.sub(r'[^\x00-\x7F]+', '', email)

    # Remove extra whitespace
    email = re.sub(r'\s+', ' ', email).strip()

    return email

In [129]:
cleaned_df['cleaned_body'] = cleaned_df['body'].apply(clean_email_with_soup)

/var/folders/m1/502kn7jx4nd1sm29zp721mh40000gn/T/ipykernel_10766/3139876248.py:16: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(email, 'html.parser')


----
**Comment:**

`MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup`

Warning could be due to BeautifulSoup thinking input may be a filename or URL rather than HTML, and so added in a if statement to say if input contains tags (<>) then to use BeautifulSoup to remove them if not pass over BeautifulSoup call.


`MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(email, 'html.parser')`

This could be due to the emails where there are email attachemnts, I have a regular expression to remove file paths and files so am choosing to ignore this error.

In [130]:
cleaned_df['cleaned_subject'] = cleaned_df['subject'].apply(clean_email_with_soup)

## Export cleaned data
----

In [131]:
# exporting df to csv, ready for next stage
cleaned_df.to_csv('../../data/cleaned_emails.csv')

## Summary
----

With the email data now cleaned, I can progress to vectorising the text data using word embeddings to better capture the context behind emails. This is important as these insights will guide me when clustering.